# Resultados S vs Mario

Este notebook resume las 50 partidas jugadas entre `Group S` y `Group mario`, alternando quien inicia. El objetivo es analizar resultado, efecto del primer jugador y costo computacional.

**Configuracion:**

- `Group S`: Trial-Based Online Policy Improvement con rollouts heuristicos, `TRIALS = 50`.
- `Group mario`: agente tactico/heuristico basado en evaluacion lineal de ventanas.
- Partidas: 50.
- Alternancia: S inicia 25 partidas y Mario inicia 25 partidas.


## Carga de datos

El archivo `s_vs_mario_50_results.csv` tiene dos filas por partida: una desde la perspectiva de S y otra desde la perspectiva de Mario. Para evitar duplicar conteos, los resumenes por partida usan solo las filas donde `focal_side == 'agent_1'`.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
CSV_PATH = PROJECT_ROOT / 's_vs_mario_50_results.csv'
FIG_DIR = PROJECT_ROOT / 'figuras_entrega'

df = pd.read_csv(CSV_PATH)
games = df[df['focal_side'] == 'agent_1'].copy().sort_values('game_id')

print('Filas del CSV:', len(df))
print('Partidas reales:', games['game_id'].nunique())
print('Tiempo total medido (min):', round(games['total_game_time'].sum() / 60, 2))

games.head()


## Resumen global

| Resultado | Partidas | Porcentaje |
| --- | ---: | ---: |
| Gana S | 38 | 76% |
| Gana Mario | 3 | 6% |
| Empate | 9 | 18% |

S tuvo un resultado global favorable: gano 38 de 50 partidas. Mario logro 3 victorias y forzo 9 empates.

In [ ]:
winner_summary = (
    games['winner_agent']
    .value_counts()
    .rename_axis('winner_agent')
    .reset_index(name='partidas')
)
winner_summary['porcentaje'] = winner_summary['partidas'] / len(games)
winner_summary


## Resultado segun quien inicia

| Quien inicia | Partidas | Gana S | Gana Mario | Empates | Win rate S |
| --- | ---: | ---: | ---: | ---: | ---: |
| Mario inicia | 25 | 13 | 3 | 9 | 52% |
| S inicia | 25 | 25 | 0 | 0 | 100% |

El orden de inicio cambia bastante el resultado. Cuando S inicia, gana todas las partidas. Cuando Mario inicia, Mario no domina, pero logra 3 victorias y 9 empates.

In [ ]:
by_starter = (
    games.groupby(['starter', 'winner_agent'])
    .size()
    .unstack(fill_value=0)
)
by_starter


## Grafica 1: resultados alternando inicio

La primera grafica muestra el resultado global y luego separa el resultado segun quien inicia.

![Resultados S vs Mario](figuras_entrega/12_s_vs_mario_resultados.png)

## Tiempos y duracion

| Metrica | Valor |
| --- | ---: |
| Tiempo total medido | 13.19 min |
| Tiempo promedio por partida | 15.83 s |
| Movimientos promedio por partida | 20.36 |

Mario es practicamente instantaneo comparado con S. La mayor parte del costo computacional viene de los rollouts heuristicos de S.

In [ ]:
timing_summary = pd.DataFrame({
    'metrica': [
        'tiempo_total_min',
        'tiempo_promedio_partida_s',
        'movimientos_promedio',
    ],
    'valor': [
        games['total_game_time'].sum() / 60,
        games['total_game_time'].mean(),
        games['num_moves'].mean(),
    ],
})
timing_summary


## Grafica 2: costo computacional

La segunda grafica compara el tiempo promedio por jugada de cada agente y muestra la duracion de cada partida.

![Tiempos S vs Mario](figuras_entrega/13_s_vs_mario_tiempos.png)

## Lectura final

S gana claramente cuando inicia y mantiene ventaja global cuando Mario inicia, aunque en ese caso aparecen mas empates. Mario es mucho mas rapido por decision, asi que la comparacion muestra un intercambio claro: S logra mejor desempeno competitivo, pero pagando mas tiempo de computo por jugada.

## Regenerar graficas

Esta celda es opcional. Solo hace falta ejecutarla si se modifica el CSV o si se quieren volver a exportar las imagenes.

In [ ]:
REGENERAR_GRAFICAS = False

if REGENERAR_GRAFICAS:
    import runpy
    runpy.run_path('generar_graficas_s_vs_mario.py')
